In [1]:
import pandas as pd
import json
from tqdm import tqdm

# ======================
# FILE PATHS
# ======================
CSV_FILE = "EASC.csv"
JSON_FILES = ["arabic_val.jsonl", "arabic_test.jsonl"]  # list of JSON files
OUTPUT_FILE = "combined_dataset.csv"  # CSV output

# ======================
# LOAD CSV DATA
# ======================
def load_csv(path):
    df = pd.read_csv(path)
    data = {"text": [], "summary": [], "id": [], "url": [], "title": []}
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Loading CSV"):
        t, s = str(r["text_x"]).strip(), str(r["text_y"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)
            data["id"].append(str(r.get("id", "")))
            data["url"].append(str(r.get("url", "")))
            data["title"].append(str(r.get("title", "")))
    return pd.DataFrame(data)

# ======================
# LOAD JSON DATA
# ======================
def load_json(path):
    data = {"text": [], "summary": [], "id": [], "url": [], "title": []}
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Loading {path}"):
            j = json.loads(line)
            if j.get("text") and j.get("summary"):
                data["text"].append(j["text"])
                data["summary"].append(j["summary"])
                data["id"].append(j.get("id", ""))
                data["url"].append(j.get("url", ""))
                data["title"].append(j.get("title", ""))
    return pd.DataFrame(data)

# ======================
# COMBINE DATA
# ======================
csv_df = load_csv(CSV_FILE)

# Load and combine all JSON files
json_dfs = [load_json(jf) for jf in JSON_FILES]
json_df = pd.concat(json_dfs, ignore_index=True)

# Combine CSV + JSON
combined_df = pd.concat([csv_df, json_df], ignore_index=True)
print(f"✅ Combined dataset size: {len(combined_df)}")

# ======================
# SAVE TO CSV
# ======================
combined_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"✅ Saved combined dataset to {OUTPUT_FILE}")


Loading CSV: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 153/153 [00:00<00:00, 16321.91it/s]
Loading arabic_val.jsonl: 4689it [00:00, 24436.85it/s]
Loading arabic_test.jsonl: 4689it [00:00, 24528.15it/s]


✅ Combined dataset size: 9531
✅ Saved combined dataset to combined_dataset.csv


In [2]:
import pandas as pd
from datasets import Dataset
from tqdm import tqdm

DATA_FILE = "combined_dataset.csv"
OUTPUT_FILE = "combined_dataset.csv"

def load_csv(path):
    data = pd.read_csv(path, encoding="utf-8")

    # Ensure columns exist
    if "text" not in data.columns or "summary" not in data.columns:
        raise ValueError("CSV must contain 'text' and 'summary' columns.")

    # Calculate text lengths (number of words)
    data["text_length_x"] = data["text"].apply(lambda x: len(str(x).split()))
    data["text_length_y"] = data["summary"].apply(lambda x: len(str(x).split()))

    # Compute medians
    median_x = data["text_length_x"].median()
    median_y = data["text_length_y"].median()
    print(f"📝 Median text length (text_x): {median_x} words")
    print(f"📝 Median summary length (text_y): {median_y} words")

    # Categorize length relative to median
    def length_category(length, median):
        if length < median * 0.75:
            return "short"
        elif length <= median * 1.25:
            return "medium"
        else:
            return "long"

    data["length_category_x"] = data["text_length_x"].apply(lambda l: length_category(l, median_x))
    data["length_category_y"] = data["text_length_y"].apply(lambda l: length_category(l, median_y))

    # Convert to Hugging Face Dataset
    data_dict = {
        "text": [], "summary": [],
        "text_length_x": [], "length_category_x": [],
        "text_length_y": [], "length_category_y": []
    }
    for _, row in tqdm(data.iterrows(), total=len(data), desc="Loading dataset"):
        data_dict["text"].append(row["text"])
        data_dict["summary"].append(row["summary"])
        data_dict["text_length_x"].append(row["text_length_x"])
        data_dict["length_category_x"].append(row["length_category_x"])
        data_dict["text_length_y"].append(row["text_length_y"])
        data_dict["length_category_y"].append(row["length_category_y"])

    dataset = Dataset.from_dict(data_dict)
    return dataset, data

# Load dataset and compute lengths
dataset, updated_df = load_csv(DATA_FILE)
print(f"📊 Dataset size: {len(dataset)}")
print(updated_df.head())
updated_df=updated_df.drop("url", axis=1)  # axis=1 means columns
updated_df=updated_df.drop("title", axis=1)  # axis=1 means columns

# Save to new CSV
updated_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"✅ Updated dataset saved as {OUTPUT_FILE}")


📝 Median text length (text_x): 314.0 words
📝 Median summary length (text_y): 23.0 words


Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9531/9531 [00:00<00:00, 20244.92it/s]


📊 Dataset size: 9531
                                                text  \
0  لودفيج فان بيتهوفن مؤلف موسيقي ألماني ولد عام ...   
1  الناي آلة نفخية تعد بحق أقدم آلة موسيقية في ال...   
2  العود هي آلة موسيقية شرقية وترية تاريخها موغل ...   
3  الخطاط الكويتي فريد العلي يعرض مجموعة محمديات ...   
4  لفت نظري واستغرابي إعلانان كلاهما يتعلق بـ"طلب...   

                                             summary id  url title  \
0  له الفضل الاعظم في تطوير الموسيقى الكلاسيكية. ...  1  NaN   NaN   
1  الناي آلة نفخية تعد بحق اقدم آلة موسيقية في ال...  2  NaN   NaN   
2  العود هي آلة موسيقية شرقية وترية تاريخها موغل ...  3  NaN   NaN   
3  وفي اليوم الثاني من المعرض اقام الفنان العلي و...  4  NaN   NaN   
4  لفت نظري واستغرابي اعلانان كلاهما يتعلق بـ"طلب...  5  NaN   NaN   

   text_length_x  text_length_y length_category_x length_category_y  
0            436             68              long              long  
1            371            231            medium              long  
2          

In [3]:
import json
import pandas as pd
from tqdm import tqdm

# ======================
# FILE PATHS
# ======================
JSON_FILES = ["arabic_val.jsonl", "arabic_test.jsonl"]  # list of JSONL files
OUTPUT_CSV = "xl_sum.csv"                      # CSV output

# ======================
# LOAD JSONL FILES AND COMBINE
# ======================
data = {"id": [], "url": [], "title": [], "summary": [], "text": []}

for JSON_FILE in JSON_FILES:
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Processing {JSON_FILE}"):
            j = json.loads(line)
            if j.get("text") and j.get("summary"):  # only keep if both exist
                data["id"].append(j.get("id", ""))
                data["url"].append(j.get("url", ""))
                data["title"].append(j.get("title", ""))
                data["summary"].append(j.get("summary", ""))
                data["text"].append(j.get("text", ""))

# ======================
# CREATE DATAFRAME AND SAVE TO CSV
# ======================
df = pd.DataFrame(data)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"✅ JSONL files combined and saved to CSV: {OUTPUT_CSV}")


Processing arabic_val.jsonl: 4689it [00:00, 27766.42it/s]
Processing arabic_test.jsonl: 4689it [00:00, 30592.02it/s]


✅ JSONL files combined and saved to CSV: xl_sum.csv


In [35]:
import pandas as pd
import csv

# Function to calculate text lengths
def add_text_lengths(df, text_col, summary_col=None):
    df["text_length_x"] = df[text_col].apply(lambda x: len(str(x)))
    if summary_col and summary_col in df.columns:
        df["text_length_y"] = df[summary_col].apply(lambda x: len(str(x)))
    else:
        df["text_length_y"] = 0  # if no summary, set 0
    return df

# Load the first CSV and take 3000 rows
df_expert = pd.read_csv("/home/kawkab/summerize/datsets/process2/scientific_subset_3000_arabic.csv").head(3000)
df_expert_new = pd.DataFrame(columns=[
    "text", "summary", "id",
    "text_length_x", "text_length_y",
    "length_category_x", "length_category_y", "type"
])
df_expert_new["text"] = df_expert["text"]
df_expert_new["summary"] = df_expert.get("summary", "")
df_expert_new["id"] = df_expert.get("id", "")
df_expert_new["type"] = "expert"
df_expert_new = add_text_lengths(df_expert_new, "text", "summary")
df_expert_new.fillna("", inplace=True)

# Load the second CSV and take 3000 rows
df_kids = pd.read_csv("/home/kawkab/summerize/datsets/arabic_stories.csv").head(3000)
df_kids_new = pd.DataFrame(columns=[
    "text", "summary", "id",
    "text_length_x", "text_length_y",
    "length_category_x", "length_category_y", "type"
])
df_kids_new["text"] = df_kids["story"]
df_kids_new["summary"] = df_kids.get("summary", "")
df_kids_new["id"] = df_kids.get("id", "")
df_kids_new["type"] = "kids"
df_kids_new = add_text_lengths(df_kids_new, "text", "summary")
df_kids_new.fillna("", inplace=True)

# Load the third CSV and take 3000 rows
df_easc = pd.read_csv("/home/kawkab/summerize/datsets/xl_sum.csv").head(3000)
df_easc_new = pd.DataFrame(columns=[
    "text", "summary", "id",
    "text_length_x", "text_length_y",
    "length_category_x", "length_category_y", "type"
])
df_easc_new["text"] = df_easc["text"]  # main text column
df_easc_new["summary"] = df_easc.get("summary", "")
df_easc_new["id"] = df_easc.get("id", "")
df_easc_new["type"] = "adult"
df_easc_new = add_text_lengths(df_easc_new, "text", "summary")
df_easc_new.fillna("", inplace=True)

# Merge all datasets
df_final = pd.concat([df_expert_new, df_kids_new, df_easc_new], ignore_index=True)

# Save to CSV
output_file = "/home/kawkab/summerize/datsets/merged_dataset.csv"
df_final.to_csv(output_file, index=False, quoting=csv.QUOTE_NONNUMERIC)

print(f"CSV file '{output_file}' created successfully with {len(df_final)} rows!")


CSV file '/home/kawkab/summerize/datsets/merged_dataset.csv' created successfully with 9000 rows!


In [6]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv('/home/kawkab/summerize/datsets/process2/scientific_subset_3000.csv')

# Use a lightweight model (faster, lower quality)
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",  # or "sshleifer/distilbart-cnn-12-6" for speed
    device=0  # Use GPU if available (remove if CPU-only)
)

def safe_summarize(text, max_len=2000):
    try:
        # Transformers need 3+ tokens and handle max 1024 tokens
        text = str(text)[:1024]
        if len(text.split()) < 5:
            return text  # Too short to summarize
        
        summary = summarizer(
            text,
            max_length=max_len,
            min_length=10,
            do_sample=False
        )[0]['summary_text']
        return summary
    except Exception as e:
        return text  # Fallback to original on error

# Apply to first 100 rows only (SLOW!)
df['text'] = df['text'].head(100).apply(safe_summarize)

df.to_csv('test_summarized.csv', index=False)

/singularity/00-gupyter2/.ve-jhub/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-04 15:38:09.954327: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Device set to use cuda:0
Your max_length is set to 100, but your input_length is only 90. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=45)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Your max_length is set to 100

KeyboardInterrupt: 

In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from arabert.preprocess import ArabertPreprocessor
import pandas as pd
from tqdm import tqdm

# -----------------------------
# Load model
# -----------------------------
model_name = "malmarjeh/t5-arabic-text-summarization"
preprocessor = ArabertPreprocessor(model_name="")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0  # set to -1 if using CPU
)

# -----------------------------
# Audience prompts (more concise for experts)
# -----------------------------
PROMPTS = {
    "child": (
        "لخص النص لطفل:\n"
        "استخرج الفكرة الرئيسية فقط.\n"
        "استخدم كلمات بسيطة.\n"
        "جمل قصيرة وواضحة.\n"
        "تجنب التفاصيل المعقدة.\n"
        "رتب الأفكار ببساطة.\n"
        "اختم برسالة إيجابية.\n\n"
    ),
    "adult": (
        "لخص النص لعامة الناس:\n"
        "وضح الفكرة الرئيسية بوضوح.\n"
        "لغة بسيطة ومحترمة.\n"
        "اشرح لماذا الموضوع مهم.\n"
        "ركز على الفائدة العملية.\n"
        "تجنب التفاصيل التقنية.\n\n"
    ),
    "expert": (
        "لخص النص للخبراء بدقة وباختصار شديد:\n"
        "حدد الفكرة الأساسية فقط.\n"
        "تجنب التفاصيل، الأمثلة، والأرقام الطويلة.\n"
        "تجنب التفاصيل، الأمثلة، والأرقام الطويلة.\n"
        "استخدام مصطلحات المجال.\n"
        "تجنب التفاصيل، الأمثلة، والأرقام الطويلة.\n"
        "ركز على النتيجة الأساسية والإضافة العلمية.\n\n"
    )
}

# -----------------------------
# Helper: split text into chunks
# -----------------------------
def chunk_text(text, max_tokens=500):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
    return chunks

# -----------------------------
# Summarization function
# -----------------------------
def summarize_row(row):
    text = preprocessor.preprocess(row["text"])
    audience = row["type"]
    prompt = PROMPTS.get(audience, PROMPTS["adult"])
    
    text_len = row["text_length_x"]

    # -----------------------------
    # Set summary length proportional to audience
    # -----------------------------
    if audience == "child":
        min_tokens = max(20, int(text_len * 0.05))
        max_tokens = max(50, int(text_len * 0.2))
    elif audience == "adult":
        min_tokens = max(30, int(text_len * 0.1))
        max_tokens = max(80, int(text_len * 0.3))
    else:  # expert
        min_tokens = max(20, int(text_len * 0.05))  # 5% minimum
        max_tokens = max(60, int(text_len * 0.15))  # 15% maximum

    # -----------------------------
    # Short texts: no chunking
    # -----------------------------
    if len(tokenizer.encode(text)) < 500:
        input_text = prompt + text
        out = summarizer(
            input_text,
            min_length=min_tokens,
            max_new_tokens=max_tokens,
            num_beams=4,
            repetition_penalty=3.0,
            length_penalty=1.5,  # stronger to reduce length
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.9
        )
        summary_text = out[0]["generated_text"]
        return pd.Series([summary_text, len(summary_text)])

    # -----------------------------
    # Long texts: hierarchical summarization
    # -----------------------------
    chunks = chunk_text(text, max_tokens=500)
    chunk_summaries = []
    for chunk in chunks:
        input_text = prompt + chunk
        out = summarizer(
            input_text,
            min_length=max(10, min_tokens // 2),
            max_new_tokens=max_tokens // 2,
            num_beams=4,
            repetition_penalty=3.0,
            length_penalty=1.5,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.9
        )
        chunk_summaries.append(out[0]["generated_text"])
    
    combined_text = " ".join(chunk_summaries)
    final_input = prompt + combined_text
    final_out = summarizer(
        final_input,
        min_length=min_tokens,
        max_new_tokens=max_tokens,
        num_beams=4,
        repetition_penalty=3.0,
        length_penalty=1.5,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9
    )
    
    summary_text = final_out[0]["generated_text"]
    return pd.Series([summary_text, len(summary_text)])

# -----------------------------
# Load dataset and run summarization
# -----------------------------
tqdm.pandas()
input_path = "/home/kawkab/summerize/datsets/merged_dataset.csv"
output_path = "/home/kawkab/summerize/datsets/merged_dataset.csv"

df = pd.read_csv(input_path).head(5)  # test with first 5 rows
df[["summary", "text_length_y"]] = df.progress_apply(summarize_row, axis=1)
df.to_csv(output_path, index=False)
print("✅ Ultra-short expert-level summarization completed successfully.")


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:06<00:00,  1.29s/it]

✅ Ultra-short expert-level summarization completed successfully.
